# knaif skill workbench

Type one utterance. See the plan the model made, the command it renders, where it ran, and
how long it took — Python, native, or both side by side.

**This is not an acceptance instrument.** It runs whatever you type: no corpus, no floors,
no safety gate. `just eval-accept` is the bar and nothing here can move it. A good result
here is a reason to run the eval suite, not a reason to publish.

**Run it, don't read it.** The selectors are `ipywidgets`, which render nothing on GitHub.
Every cell works when re-run top to bottom after a kernel restart.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks" / "shared"))

from workbench import inventory, panel, selectors
from workbench.runners import NativeRunner, PythonRunner

# Fixtures are COPIED in here. sandbox/fixtures/ is never written to — a workbench that
# overwrote it would quietly poison every later eval run.
SCRATCH = ROOT / "sandbox" / "workbench"
SCRATCH.mkdir(parents=True, exist_ok=True)

inv = inventory.scan(ROOT)
print(inventory.summary(inv))

## Pick what to run

Three axes, each scoped to where it is real. **Compute** is a *build* picker because CUDA
versus Vulkan is compiled in, not switched at run time — and each build is labelled by what
it reports (`backend list --json`), never by its directory name.

**force CPU** is the one control that moves both runtimes.

In [ ]:
sel = selectors.build_widgets(inv).selection

## Run one utterance

In [ ]:
UTTERANCE = "convert clip.mp4 to mkv"

results = []
if sel.wants_python:
    agent = selectors.python_agent(sel, root=ROOT, sandbox=SCRATCH)
    results.append(
        PythonRunner(agent, skill=sel.skill, work_dir=SCRATCH).run(UTTERANCE, dry_run=sel.dry_run)
    )
if sel.wants_native and sel.build is not None:
    results.append(
        NativeRunner(
            binary=sel.build.path,
            model_path=ROOT / sel.model.path,
            skill=sel.skill,
            work_dir=SCRATCH,
            force_cpu=sel.force_cpu,
            backends_dir=sel.backends_dir,
        ).run(UTTERANCE, dry_run=sel.dry_run)
    )

print(sel.describe())
for r in results:
    print()
    print(panel.show(r))

## Side by side

Only meaningful with **Runtime = Both**. Native is `in-progress` for every skill, so a
native disagreement is informative — it must not block a Python decision.

In [ ]:
if len(results) == 2:
    print(panel.compare(*results))
else:
    print("set Runtime = Both to compare")

## Repeat it

One sample is a number, not a measurement. Every percentile reported is a figure some run
actually took.

In [ ]:
N = 5
runner = PythonRunner(agent, skill=sel.skill, work_dir=SCRATCH) if sel.wants_python else None
if runner is not None:
    repeated = [runner.run(UTTERANCE, dry_run=sel.dry_run) for _ in range(N)]
    print(panel.stats(repeated))

---
A good result here is a reason to run the eval suite. It is not a reason to publish.